In [ ]:
import pandas as pd
import os
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# normal_directory = "Normal"
# attack_directory = "Attack"

# # Create lists to store normal and attack DataFrames
# normal_dataframes = []
# attack_dataframes = []

# # Loop through normal traffic files
# for filename in os.listdir(normal_directory):
#     if filename.endswith(".csv"):
#         file_path = os.path.join(normal_directory, filename)
#         df = pd.read_csv(file_path)
#         normal_dataframes.append(df)

# # Loop through attack traffic files
# for filename in os.listdir(attack_directory):
#     if filename.endswith(".csv"):
#         file_path = os.path.join(attack_directory, filename)
#         df = pd.read_csv(file_path)
#         attack_dataframes.append(df)

# # Concatenate normal and attack DataFrames separately
# normal_concatenated_df = pd.concat(normal_dataframes, ignore_index=True)
# attack_concatenated_df = pd.concat(attack_dataframes, ignore_index=True)

# # Assign labels to the concatenated DataFrames
# normal_concatenated_df['label'] = 'normal'
# attack_concatenated_df['label'] = 'attack'

# # Combine normal and attack DataFrames into a single DataFrame
# combined_df = pd.concat([normal_concatenated_df, attack_concatenated_df], ignore_index=True)

In [ ]:
# combined_df.to_csv('combined_df.csv')

In [ ]:
combined_df=pd.read_csv('/kaggle/input/finaldataset/final.csv')

In [ ]:
combined_df['label'].value_counts()

In [ ]:
combined_df[combined_df['label']=='attack']

In [ ]:
import matplotlib.pyplot as plt
combined_df['label'].value_counts().plot(kind='bar')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
ax = combined_df['Protocol'].value_counts().plot(kind='bar')
plt.xlabel('Protocol')
plt.ylabel('Count')
plt.title('Protocol Distribution')

# Adding count numbers on top of each bar
for p in ax.patches:
    ax.annotate(str(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points')

plt.show()

In [ ]:
import ipaddress
import re

def convert_address_to_numeric(address):
    if ':' in address:
        # MAC address
        mac_address = re.sub(r':', '', address)
        return int(mac_address, 16)
    else:
        # IPv4 address
        try:
            return int(ipaddress.IPv4Address(address))
        except ValueError:
            # Handle non-IPv4 addresses
            return address


In [ ]:
combined_df = combined_df[(combined_df['Source'].apply(lambda x: '.' in x or ':' in x)) &
                          (combined_df['Destination'].apply(lambda x: '.' in x or ':' in x))]

# Apply the convert_address_to_numeric function
combined_df['Source'] = combined_df['Source'].apply(convert_address_to_numeric)
combined_df['Destination'] = combined_df['Destination'].apply(convert_address_to_numeric)

In [ ]:
combined_df = combined_df.drop(['No.','Info'], axis=1)

In [ ]:
combined_df = combined_df.drop('Unnamed: 0', axis=1)

In [ ]:
combined_df

In [ ]:
combined_df['Protocol'] = combined_df['Protocol'].astype('category')

In [ ]:
protocol_dummies = pd.get_dummies(combined_df['Protocol'], prefix='Protocol')
combined_df = pd.concat([combined_df, protocol_dummies.astype(int)], axis=1)
# combined_df = combined_df.drop('Protocol', axis=1)



In [ ]:
combined_df = combined_df.drop('Protocol', axis=1)

In [ ]:
combined_df.columns

In [ ]:
combined_df

In [ ]:
combined_df['Source'] = combined_df['Source'].astype(float)

In [ ]:
combined_df

In [ ]:
# from imblearn.under_sampling import NearMiss

# #Split the data into features and target
# X = combined_df.drop('label', axis=1)
# y = combined_df['label']

# from imblearn.under_sampling import RandomUnderSampler

# undersampler = RandomUnderSampler()
# X_resampled, y_resampled = undersampler.fit_resample(X, y)

# # Convert the resampled data back to a DataFrame
# data_resampled = pd.concat([X_resampled, y_resampled], axis=1)

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape

In [ ]:
X_train.info()

In [ ]:
y_train

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.fit_transform(y_test)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create PyTorch datasets and data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class CNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * (input_size // 4), 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add a channel dimension
        x = self.pool(nn.functional.relu(self.conv1(x)))
        x = self.pool(nn.functional.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
input_size = X_train.shape[1]
num_classes = 2  # Binary classification (attack or normal)
model = CNN(input_size, num_classes)

# Define the loss function and optimizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


In [ ]:
from tqdm import tqdm

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, num_epochs, device):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        running_correct = 0
        total = 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for inputs, labels in progress_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            running_correct += (predicted == labels).sum().item()

            progress_bar.set_postfix(loss=running_loss / (len(progress_bar) + 1),
                                     accuracy=running_correct / total * 100)

        # Compute validation accuracy
        val_correct = 0
        val_total = 0
        model.eval()
        with torch.no_grad():
            for val_inputs, val_labels in val_loader:
                val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
                val_outputs = model(val_inputs)
                _, val_predicted = torch.max(val_outputs.data, 1)
                val_total += val_labels.size(0)
                val_correct += (val_predicted == val_labels).sum().item()
        
        val_accuracy = val_correct / val_total * 100
        model.train()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = running_correct / total * 100
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%, Val Accuracy: {val_accuracy:.2f}%')

    return model

In [ ]:
def save_checkpoint(model, optimizer, epoch, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }, path)

# Set device to GPU if available, else CPU
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# =model = model.to(device)

In [ ]:
num_epochs = 10

trained_model = train_model(model, train_loader,test_loader ,optimizer, criterion, num_epochs, device)

# Save the model checkpoint
save_checkpoint(trained_model, optimizer, num_epochs, '/kaggle/working/model_checkpoint.pth')

In [ ]:
input_size= X_train.shape[1]

In [ ]:
input_size

In [ ]:
loaded_model = CNN(32, 2)
loaded_model.load_state_dict(torch.load('/kaggle/working/model.pth'))
loaded_model.eval()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_model =loaded_model.to(device)

In [ ]:
loaded_model.eval()  # Set the model to evaluation mode
predictions = []

with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)  # Move inputs to the appropriate device
        outputs = loaded_model(inputs)
        _, predicted_classes = torch.max(outputs, 1)
        predictions.extend(predicted_classes.cpu().numpy().tolist())



In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

labels = np.array(y_test_tensor)
predictions = np.array(predictions)

# Calculate confusion matrix
conf_matrix = confusion_matrix(labels, predictions)

# Calculate classification report
class_report = classification_report(labels, predictions)

print("Confusion Matrix:")
print(conf_matrix)

print("\nClassification Report:")
print(class_report)

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()